In [ ]:
# Less Salt Less Sugar Monthly Report (launch from 31Aug2026. last for 1 year at least)
#     1. Daily Badge impression by device 
#     2. Daily Brand Page (LMS) Pageview/ Impressions   
#     3. Daily Theme Listing Impressions (adv search)

#http://192.168.61.59:8888/notebooks/reports/monthly_icon_impression_report/monthly_icon_impression_report_result.ipynb

# $: 142.24x3=426.72GB
# 1TB $50, 426.72GB ≈ $21.34

In [2]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../../BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [3]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month

all_dates = pd.date_range(
    start=f"{year}-{str_month}-01",
    end=pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0),
    freq="D",
)

str_month

'09'

In [ ]:
template_file = 'Pet_report_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

# excel_name = "Pet_report_2026-10-05.xlsx"

In [ ]:
#  1. Quick Search
sql_1 = f"""
SELECT
    DATE(time) AS querydate, platform, COUNT(1) AS count
FROM `openrice-production.ORGA.PV_{year}{str_month}*`
WHERE
    (
        LOWER(platform) IN ('desktop', 'mobile')        -- Web
        AND LOWER(EventAction) IN ('view.sr1.promotion')
        AND LOWER(EventLabelRaw) LIKE '%promotionid:100%'
        AND LOWER(EventLabelRaw) LIKE '%amenityid=20232%'
    )
    OR
    (
        LOWER(platform) IN ('android', 'ios', 'hms')        -- App
        AND LOWER(EventAction) IN ('or.search.quick')
        AND (
             (  LOWER(EventLabelRaw) LIKE '%amenityid:20232%' and LOWER(EventLabelRaw) LIKE '%sn:hk.qcksearch%'  )
        )
    )
GROUP BY querydate, platform
"""

df_bq_1 = client.query(sql_1).result().to_dataframe()

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
pivoted_df_1 = df_bq_1.pivot(index='querydate', columns='platform', values='count')
pivoted_df_1 = pivoted_df_1.reindex(index=all_dates, fill_value=0).fillna(0)

expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_1 = pivoted_df_1.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_1["Web"]=pivoted_df_1["desktop"] + pivoted_df_1["mobile"]
pivoted_df_1["App"]=pivoted_df_1["ios"] + pivoted_df_1["android"] + pivoted_df_1["hms"]
pivoted_df_1 =pivoted_df_1[["Web","App"]]

pivoted_df_1.index.name = None
pivoted_df_1.reset_index(inplace=True)
pivoted_df_1["index"] = pivoted_df_1["index"].dt.date
pivoted_df_1

platform,index,Web,App
0,2026-09-01,142,129
1,2026-09-02,111,198
2,2026-09-03,93,193
3,2026-09-04,90,152
4,2026-09-05,55,230
5,2026-09-06,49,227
6,2026-09-07,113,167
7,2026-09-08,76,213
8,2026-09-09,115,163
9,2026-09-10,96,177


In [ ]:
# group by platform but not date
df_bq_1_platform = df_bq_1.groupby('platform', as_index=False)['count'].sum()
df_bq_1_platform

,platform,count
0,android,2080
1,desktop,1578
2,hms,73
3,ios,3033
4,mobile,799


In [ ]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df_1.to_excel(writer, sheet_name='sheet1', startrow=2, header=None, index=False)

In [ ]:
# 2. Adv Search

sql_2 = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE    
        (   
            LOWER(platform) IN ('desktop', 'mobile')           -- web
            and LOWER(EventAction) = 'or.search.adv'
            and LOWER(EventLabelRaw) LIKE '%conditionid:11024%'
            and LOWER(EventLabelRaw) LIKE '%amenityid=20232%'
        )  
        OR   
        (   LOWER(platform) in ('android','ios','hms')         -- app
            and LOWER(EventAction) = 'or.search.adv'
            and LOWER(EventLabelRaw) LIKE '%featurefit:11024%'
            and LOWER(EventLabelRaw) LIKE '%sn:hk.advsearch%'    
        )  

    group by platform, querydate
    """

df_bq_2 = client.query(sql_2).result().to_dataframe()

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
#df_bq_2

In [ ]:
pivoted_df_2 = df_bq_2.pivot(index='querydate', columns='platform', values='count')
pivoted_df_2 = pivoted_df_2.reindex(index=all_dates, fill_value=0).fillna(0)
expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_2 = pivoted_df_2.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_2["Web"]= pivoted_df_2["desktop"] + pivoted_df_2["mobile"]
pivoted_df_2["App"]=pivoted_df_2["ios"] + pivoted_df_2["android"] + pivoted_df_2["hms"]
pivoted_df_2 =pivoted_df_2[["Web","App"]]

pivoted_df_2.index.name = None
pivoted_df_2.reset_index(inplace=True)
pivoted_df_2["index"] = pivoted_df_2["index"].dt.date
pivoted_df_2

platform,index,Web,App
0,2026-09-01,17,30
1,2026-09-02,6,37
2,2026-09-03,4,32
3,2026-09-04,6,34
4,2026-09-05,2,35
5,2026-09-06,9,30
6,2026-09-07,13,55
7,2026-09-08,4,29
8,2026-09-09,1,32
9,2026-09-10,7,25


In [ ]:
with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df_2.to_excel(writer, sheet_name='sheet1', startrow=37, header=None, index=False)

Below is Testing!!!

is Testing!!!

Testing!!!

In [ ]:
# Theme Listing
# and STRPOS(LOWER(COALESCE(EventLabelRaw, '')),'promotionid:100') > 0 --> contain %

sql_test =f"""

select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_20261005`
    WHERE    
        (   
            LOWER(platform) IN ('desktop', 'mobile')           -- web
            and LOWER(EventAction) = 'view.sr1.promotion'
            and STRPOS(LOWER(COALESCE(EventLabelRaw, '')),'promotionid:13,13,13,13,13,13,13,13,13,13,13,13,13,13,13') > 0
            and STRPOS(LOWER(COALESCE(EventLabelRaw, '')),'/restaurants/type/') > 0 
        )  
        OR   
        (   LOWER(platform) in ('android','ios','hms')           --app
            and LOWER(EventAction) = 'or.search.adv'
            and LOWER(EventLabelRaw) LIKE '%35336%'    
        )  

    group by platform, querydate

"""
df_bq_test = client.query(sql_test).result().to_dataframe()
df_bq_test


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,querydate,platform,count
0,2026-10-05,android,10
1,2026-10-05,mobile,1
2,2026-10-05,desktop,6
3,2026-10-05,ios,24


In [13]:
expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_test = (
    df_bq_test
    .pivot(index="querydate", columns="platform", values="count")
    .reindex(columns=expected_platforms, fill_value=0)
    .fillna(0)
)
pivoted_df_test["Web"] = (pivoted_df_test["desktop"] + pivoted_df_test["mobile"])
pivoted_df_test["App"] = (pivoted_df_test["ios"]+ pivoted_df_test["android"]+ pivoted_df_test["hms"])

pivoted_df_test = pivoted_df_test[["Web", "App"]].reset_index()
pivoted_df_test["querydate"] = pd.to_datetime(pivoted_df_test["querydate"]).dt.date

pivoted_df_test

platform,querydate,Web,App
0,2026-10-05,7,34


In [25]:
# brand page

sql_test_1 = f"""
SELECT
    DATE(time) AS querydate, platform, COUNT(1) AS count
FROM `openrice-production.ORGA.PV_20261005`
WHERE
    (
        LOWER(platform) IN ('desktop', 'mobile')        -- Web
        AND LOWER(EventAction) IN ('view.sr1.promotion', 'or.search.adv')
        AND LOWER(EventLabelRaw) LIKE '%35336%'
    )
    OR
    (
        LOWER(platform) IN ('android', 'ios', 'hms')        -- App
        AND LOWER(EventAction) IN ('or.search.layer.search', 'or.search.quick')
        AND (
            LOWER(COALESCE(EventLabelRaw, '')) LIKE '%35336%'
            OR 
            LOWER(COALESCE(EventLabelRaw, '')) LIKE '%少鹽少糖%'
        )
    )
GROUP BY querydate, platform
"""

df_bq_test_1 = client.query(sql_test_1).result().to_dataframe()
df_bq_test_1

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,querydate,platform,count
0,2026-10-05,android,45
1,2026-10-05,desktop,51
2,2026-10-05,hms,5
3,2026-10-05,ios,50
4,2026-10-05,mobile,20


In [30]:
expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_test_1 = (
    df_bq_test_1
    .pivot(index="querydate", columns="platform", values="count")
    .reindex(columns=expected_platforms, fill_value=0)
    .fillna(0)
)
pivoted_df_test_1["Web"] = (pivoted_df_test_1["desktop"] + pivoted_df_test_1["mobile"])
pivoted_df_test_1["App"] = (pivoted_df_test_1["ios"]+ pivoted_df_test_1["android"]+ pivoted_df_test_1["hms"])

pivoted_df_test_1 = pivoted_df_test_1[["Web", "App"]].reset_index()
pivoted_df_test_1["querydate"] = pd.to_datetime(pivoted_df_test_1["querydate"]).dt.date

pivoted_df_test_1

platform,querydate,Web,App
0,2026-10-05,71,100
